# 5. 课堂练习

> 三个练习，从「能跑通」到「能观测」再到「能兜底」，对应多 Agent 系统的三种核心能力：**编排、并发、治理**。

## 练习概览

| 练习 | 难度 | 主题 | 核心考点 | 依赖 |
| --- | --- | --- | --- | --- |
| A | ⭐ | 三角色流水线 | 角色定义、DAG 编排、终止条件 | 无（本页自带客户端） |
| B | ⭐⭐ | 消息队列 Pipeline | 队列解耦、并发、观测指标 | 无（标准库） |
| C | ⭐⭐⭐ | Supervisor + HITL | 质量阈值、人工审查、决策记录 | 06 的 4.3 节 |

## 建议顺序

1. 先跑通**练习 A**，确认 API 与模型可用
2. 再做**练习 B**，把「串行调用」改成「队列驱动」
3. 最后做**练习 C**，把 Review 分支变成真正的治理流程

> 💡 目标不是把所有功能做完，而是**把架构思路跑通**：先小后大，先稳后快。


> ⚠️ **前置要求**
>
> 1. 已运行 [`03-环境初始化与MAS基础.ipynb`](03-环境初始化与MAS基础.ipynb) 安装依赖
> 2. 项目根目录 `.env` 中已配置 `OPENROUTER_API_KEY`

## 执行顺序

| 练习 | 需要先运行的内容 |
| --- | --- |
| 练习 A | 无，本页 Step 1 自行初始化客户端 |
| 练习 B | 无（只用 Python 标准库） |
| 练习 C | [`06-agentic_rag与框架对比.ipynb`](06-agentic_rag与框架对比.ipynb) 的 **4.3 节 Step 1 – Step 8**（同一 kernel） |

> 练习 C 复用了 06 中的 `AgenticRAGState` 与节点函数；代码里有前置检查，缺少时会给出提示。


## 5.1 练习 A（入门）：三角色流水线

### 题目

用 AutoGen **GraphFlow** 实现「给定主题生成 3 段式报告」的三角色流水线。

```mermaid
graph LR
    T([主题]) --> R[Researcher<br/>提炼要点]
    R --> W[Writer<br/>写成 3 段式]
    W --> E[Editor<br/>审校并输出 APPROVED]
```

### 要求

1. 每个 Agent 都有清晰的 `system_message`（角色、输出格式、边界）
2. 用 `DiGraphBuilder` 显式声明执行顺序
3. 用 `TextMentionTermination('APPROVED')` 作为终止条件，并用 `MaxMessageTermination` 兜底
4. 能处理任意主题输入

### 评分点

| 维度 | 看什么 |
| --- | --- |
| 角色定义 | `system_message` 是否规定了输出格式 |
| 编排 | 执行顺序是否真的是 R → W → E |
| 终止 | 除文本匹配外，是否有最大轮次保护 |
| 观测 | 能否看到每条消息的 `source` 与内容 |

> 提示：先让流程跑通，再调 prompt。三行 prompt 写不清角色，跑十次也没用。


### Step 1 — 准备 AutoGen 客户端

通过 OpenRouter 初始化两个客户端：

- **Sonnet** — 写作用
- **Haiku** — 研究与审校用，快且便宜


In [ ]:
import os
import warnings

warnings.filterwarnings('ignore')

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

if not OPENROUTER_API_KEY:
    raise OSError(
        'OPENROUTER_API_KEY 未设置。\n'
        '请前往 https://openrouter.ai 获取 Key，'
        '并在 .env 中添加：OPENROUTER_API_KEY=sk-or-v1-...'
    )

MODEL_SONNET = 'anthropic/claude-sonnet-4-5'
MODEL_HAIKU = 'anthropic/claude-haiku-4-5'

# ── AutoGen 公共导入 ────────────────────────────────────────────
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_agentchat.ui import Console
from autogen_core.models import ModelFamily, ModelInfo
from autogen_ext.models.openai import OpenAIChatCompletionClient


def make_model_info() -> ModelInfo:
    """OpenRouter 上的 Claude：显式声明能力（新版 AutoGen 必需）"""
    return {
        'vision': True,
        'function_calling': True,
        'json_output': False,
        'structured_output': False,
        'family': ModelFamily.CLAUDE_4_SONNET,
        'multiple_system_messages': True,
    }


sonnet_client = OpenAIChatCompletionClient(
    model=MODEL_SONNET,
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    model_info=make_model_info(),
)

haiku_client = OpenAIChatCompletionClient(
    model=MODEL_HAIKU,
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    model_info=make_model_info(),
)

print('✅ 环境与 AutoGen 客户端就绪')


### Step 2 — 补全 TODO


In [ ]:
# ============================================================
# 练习 A — 在 TODO 处补全代码
# ============================================================
# 对应 5.1 的四条要求：
#   1. 三个 Agent 都要有清晰的 system_message
#   2. 用 DiGraphBuilder 连成 Researcher → Writer → Editor
#   3. 终止条件 = TextMentionTermination('APPROVED') | MaxMessageTermination(10)
#   4. 能处理任意主题

# TODO 1: Researcher —— 输出 3-5 个要点，每点以「- 」开头
ex_researcher = None

# TODO 2: Writer —— 把要点扩写成 3 段式文章（引言 / 主体 / 结论）
ex_writer = None

# TODO 3: Editor —— 审校后输出 APPROVED
ex_editor = None

# TODO 4: 用 DiGraphBuilder 构建 Researcher → Writer → Editor

# TODO 5: 组装 GraphFlow（含终止条件）

# TODO 6: 运行（取消注释）
# await Console(my_flow.run_stream(task='请写一篇关于「强化学习」的报告'))

print('📝 完成 TODO 1-6 后，取消注释最后一行运行')


### Step 3 — 参考答案

> 先自己写，再对照。重点看**为什么这样分工**，而不是逐字抄 prompt。


In [ ]:
# 参考答案：选中本 cell 后取消注释（VS Code: Cmd/Ctrl + /）再运行
#
# ans_researcher = AssistantAgent(
#     name='Researcher',
#     model_client=haiku_client,
#     system_message=(
#         '你是研究员。收到主题后输出 3-5 个关键要点，'
#         '每个要点以「- 」开头，不要写标题。'
#     ),
# )
#
# ans_writer = AssistantAgent(
#     name='Writer',
#     model_client=sonnet_client,
#     system_message=(
#         '你是写作者。根据要点撰写 3 段式文章：引言、主体、结论。'
#         '语言流畅，面向普通读者。'
#     ),
# )
#
# ans_editor = AssistantAgent(
#     name='Editor',
#     model_client=haiku_client,
#     system_message=(
#         '你是审校者。指出最多 3 处可改进之处，最后单独输出「APPROVED」。'
#     ),
# )
#
# builder = DiGraphBuilder()
# builder.add_node(ans_researcher).add_node(ans_writer).add_node(ans_editor)
# builder.add_edge(ans_researcher, ans_writer)
# builder.add_edge(ans_writer, ans_editor)
#
# flow_a = GraphFlow(
#     participants=[ans_researcher, ans_writer, ans_editor],
#     graph=builder.build(),
#     termination_condition=(
#         TextMentionTermination('APPROVED') | MaxMessageTermination(10)
#     ),
# )
#
# await Console(flow_a.run_stream(task='请写一篇关于「强化学习」的报告'))

print('💡 参考答案已注释，取消注释后运行')


## 5.2 练习 B（进阶）：消息队列 Pipeline

### 题目

用 `queue.Queue` 模拟消息队列，把串行调用改成 Pipeline：

```mermaid
graph LR
    I([Input]) --> Q1[(Q1)]
    Q1 --> R[RetrieverWorker]
    R --> Q2[(Q2)]
    Q2 --> G[GeneratorWorker]
    G --> Q3[(Q3)]
    Q3 --> V[VerifierWorker]
    V --> O([Output])
```

每个 Worker 只认识自己的**输入队列**和**输出队列**，彼此不直接调用。

### 要求

1. 实现三个 Worker 的 `process()`，每个只做本环节的事
2. Worker 之间**只通过队列通信**（禁止直接传对象）
3. 支持并发处理多条查询，并统计完成数

### 观察重点

| 指标 | 怎么测 | 说明 |
| --- | --- | --- |
| 队列长度 | `q.qsize()` | 判断是否有积压 |
| 端到端延迟 | 入队到出队的时间差 | 找出最慢的 stage |
| Worker 健康 | `t.is_alive()` | 线程是否提前退出 |
| 成功率 | `passed=True` 的占比 | 质量门效果 |

> 提示：本练习用标准库 `queue` 即可。换成 Redis / Kafka 时，只是把 `Queue` 换成外部服务，Worker 逻辑不变。


### Step 1 — 基础结构（Message 与 Worker 基类）

`BaseWorker.run()` 已经写好：阻塞取消息 → 处理 → 投递下游，收到 `None` 就向下游传递停止信号。

你只需要关心 `process()`。


In [ ]:
import queue
import threading
import time
from dataclasses import dataclass, field
from typing import Any


@dataclass
class Message:
    job_id: str
    stage: str
    payload: dict[str, Any] = field(default_factory=dict)


class BaseWorker:
    """只认识 in_queue / out_queue 的 Worker 基类。

    约定：向 out_queue 放入 None 表示「上游已结束」。
    """

    def __init__(self, name: str, in_queue: queue.Queue, out_queue: queue.Queue):
        self.name = name
        self.in_queue = in_queue
        self.out_queue = out_queue

    def process(self, message: Message) -> Message:
        raise NotImplementedError

    def run(self) -> None:
        while True:
            msg = self.in_queue.get()      # 阻塞等待：上游慢也不会提前退出
            if msg is None:                # 停止信号，继续向下游传递
                self.out_queue.put(None)
                break
            print(f'  [{self.name}] 处理 {msg.job_id}')
            self.out_queue.put(self.process(msg))


# 练习用的小型知识库
KNOWLEDGE_BASE = [
    'AutoGen 支持 GraphFlow、SelectorGroupChat 等多种协作模式。',
    'CrewAI 通过 Agent、Task、Crew 三个核心概念实现多 Agent 协作。',
    'LangGraph 使用有向图表达 Agent 工作流，支持循环、分支与状态管理。',
    'Agentic RAG 将检索拆解为粗检索、精排、生成、验证四个阶段。',
    'HITL 将低置信度结果路由到人工审查，平衡自动化与准确性。',
]

print('✅ Message / BaseWorker / 知识库就绪')


### Step 2 — 实现三个 Worker

每个 `process()` 只做一件事，并且**只读写 `message.payload` 里属于自己的字段**。


In [ ]:
class RetrieverWorker(BaseWorker):
    """TODO 1：按关键词命中数检索，把 top-3 文档放进 payload['docs']。"""

    def process(self, message: Message) -> Message:
        time.sleep(0.05)
        query = message.payload['query']

        # TODO: 用 KNOWLEDGE_BASE 做关键词匹配并排序
        docs = ['占位文档']

        return Message(message.job_id, 'retrieved', {**message.payload, 'docs': docs})


class GeneratorWorker(BaseWorker):
    """TODO 2：基于 payload['docs'] 生成答案。先写占位实现，再换 LLM。"""

    def process(self, message: Message) -> Message:
        time.sleep(0.05)

        # TODO: 拼接 docs 作为上下文，调用 LLM 生成答案
        answer = '占位答案'

        return Message(message.job_id, 'generated', {**message.payload, 'answer': answer})


class VerifierWorker(BaseWorker):
    """TODO 3：校验答案是否被 docs 支撑，结果放进 payload['passed']。"""

    def process(self, message: Message) -> Message:
        time.sleep(0.05)

        # TODO: 判断答案与文档的一致性（可先用关键词覆盖率近似）
        passed = True

        return Message(message.job_id, 'verified', {**message.payload, 'passed': passed})


print('📝 完成 TODO 1-3 后，运行下一个 cell')


### Step 3 — 运行与观测

`q1.put(None)` 是停止信号，会沿队列链依次传递到最下游。


In [ ]:
q1, q2, q3, q_out = queue.Queue(), queue.Queue(), queue.Queue(), queue.Queue()

workers = [
    RetrieverWorker('Retriever', q1, q2),
    GeneratorWorker('Generator', q2, q3),
    VerifierWorker('Verifier', q3, q_out),
]
threads = [threading.Thread(target=w.run, daemon=True) for w in workers]
for t in threads:
    t.start()

test_queries = [
    'AutoGen 的主要特性是什么？',
    'CrewAI 如何处理任务分配？',
    '什么是 Agentic RAG？',
]

print('📨 发送查询...')
for i, query in enumerate(test_queries):
    q1.put(Message(f'job_{i + 1}', 'input', {'query': query}))
q1.put(None)        # 停止信号：沿队列链依次传递

results = []
start = time.time()
print('\n📊 结果：')
while len(results) < len(test_queries):
    try:
        msg = q_out.get(timeout=30)
    except queue.Empty:
        print('  ⏰ 等待超时，检查 Worker 是否已实现')
        break
    if msg is None:
        break
    results.append(msg)
    status = '✅' if msg.payload.get('passed') else '❌'
    print(f'  {status} {msg.job_id}: {msg.payload.get("query", "")[:40]}')

# 清掉尾部的停止信号，避免被误读成「积压」
if len(results) == len(test_queries) and not q_out.empty():
    q_out.get()

elapsed = time.time() - start
print(f'\n完成：{len(results)}/{len(test_queries)} 条，耗时 {elapsed:.2f}s')
print('队列积压：', {'q1': q1.qsize(), 'q2': q2.qsize(), 'q3': q3.qsize(), 'out': q_out.qsize()})
print('Worker 存活：', {w.name: t.is_alive() for w, t in zip(workers, threads)})


## 5.3 练习 C（挑战）：Supervisor + HITL

### 题目

在 Agentic RAG 之上加一层 **Supervisor**：当最近窗口内的通过率 `pass_rate < 0.8` 时，把结果送入人工审查队列。

```mermaid
graph LR
    V[verifier] --> S{Supervisor<br/>record pass/fail}
    S -->|pass_rate ≥ 0.8| A[accept]
    S -->|pass_rate < 0.8| R[review_queue]
    R --> H[人工决策]
    H --> D[decisions 记录]
```

### 要求

1. `should_review()` — 滑动窗口内的 `pass_rate` 低于阈值时返回 `True`
2. `run_hitl_review()` — 用 `input()` 逐条收集人工决策
3. 把 Supervisor 接进 LangGraph 的条件边（替换 `orchestrator_route`）
4. 记录每条决策，便于后续分析与样本回流

### 评分点

| 维度 | 看什么 |
| --- | --- |
| 阈值逻辑 | 窗口满 / 未满时行为是否一致 |
| HITL | 是否真的等待人工输入，而不是写死 `'a'` |
| 可追溯 | `decisions` 是否包含 `query` + `decision` |
| 可回退 | 被拒绝的结果是否有后续处理路径 |

> 本练习复用 06 的 4.3 节节点函数，请先在同一 kernel 中运行 06 的 **4.3 节 Step 1 – Step 8**。


### Step 1 — 实现 Supervisor


In [ ]:
from collections import deque


class Supervisor:
    """滑动窗口统计通过率，低于阈值时触发人工审查。"""

    def __init__(self, window_size: int = 5, threshold: float = 0.8):
        self.window_size = window_size
        self.threshold = threshold
        self._recent: deque = deque(maxlen=window_size)
        self.review_queue: list = []
        self.decisions: list = []

    def record(self, passed: bool) -> None:
        """记录一次校验结果。"""
        self._recent.append(passed)

    @property
    def pass_rate(self) -> float:
        """窗口内通过率；无样本时视为 1.0（不拦截）。"""
        return sum(self._recent) / len(self._recent) if self._recent else 1.0

    def should_review(self) -> bool:
        # TODO 1: pass_rate < threshold 时返回 True
        return False

    def add_to_queue(self, item: dict) -> None:
        self.review_queue.append(item)
        print(f'  [Supervisor] 加入审查队列 (pass_rate={self.pass_rate:.2f})')

    def run_hitl_review(self) -> None:
        """逐条人工审查。注意：会阻塞等待 input()。"""
        if not self.review_queue:
            print('[Supervisor] 审查队列为空')
            return

        print(f'\n🔍 人工审查（{len(self.review_queue)} 条待审）')
        for item in self.review_queue[:]:
            print(f'  查询：{item.get("query")}')
            print(f'  草稿：{item.get("draft_answer", "")[:80]}...')
            print(f'  问题：{item.get("issues", [])}')

            # TODO 2: decision = input('批准(a)/拒绝(r): ').strip().lower()
            decision = 'a'          # 占位实现，替换为 input()

            self.decisions.append({
                'query': item['query'],
                'decision': 'approved' if decision == 'a' else 'rejected',
            })
            self.review_queue.remove(item)

        print(f'审查完成，{len(self.decisions)} 条决策已记录')


# 测试：3/5 通过 = 60% < 80%
sv = Supervisor(window_size=5, threshold=0.8)
for result in [True, True, False, False, False]:
    sv.record(result)

print(f'pass_rate = {sv.pass_rate:.2f}')
print(f'should_review() = {sv.should_review()}  ← 实现 TODO 1 后应为 True')


### Step 2 — 接入 Agentic RAG

用 `supervised_route` 替换 06 里的 `orchestrator_route`：先记录本次结果，再判断是否要人工介入。


In [ ]:
# 前置检查：本练习复用 06 中定义的共享状态与节点函数
required = [
    'AgenticRAGState',
    'retriever_node', 'reranker_node', 'generator_node', 'verifier_node',
    'accept_node', 'review_node',
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f'缺少 {", ".join(missing)}。\n'
        '请先在同一 kernel 中运行 06-agentic_rag与框架对比.ipynb 的 Step 1 – Step 8。'
    )


supervisor = Supervisor(window_size=5, threshold=0.8)


def supervised_route(state: AgenticRAGState) -> str:
    """先记录本次校验结果，再判断是否要人工介入。"""
    result = state.get('verify_result', {})
    passed = result.get('pass', True)
    supervisor.record(passed)

    if not passed or supervisor.should_review():
        supervisor.add_to_queue({
            'query': state['query'],
            'draft_answer': state['draft_answer'],
            'issues': result.get('issues', []),
        })
        return 'review'
    return 'accept'


# 构建带 Supervisor 的图（节点函数直接复用 06）
sv_builder = StateGraph(AgenticRAGState)
for name, fn in [
    ('retriever', retriever_node),
    ('reranker', reranker_node),
    ('generator', generator_node),
    ('verifier', verifier_node),
    ('accept', accept_node),
    ('review', review_node),
]:
    sv_builder.add_node(name, fn)

sv_builder.add_edge(START, 'retriever')
sv_builder.add_edge('retriever', 'reranker')
sv_builder.add_edge('reranker', 'generator')
sv_builder.add_edge('generator', 'verifier')
sv_builder.add_conditional_edges(
    'verifier',
    supervised_route,
    {'accept': 'accept', 'review': 'review'},
)
sv_builder.add_edge('accept', END)
sv_builder.add_edge('review', END)

sv_rag = sv_builder.compile()
print('✅ 带 Supervisor 的图构建完成')


# 运行两条查询，观察 pass_rate 的变化
for query in ['如何避免多 Agent 系统中的无限循环？', 'OpenRouter 有什么优势？']:
    print(f'\n❓ {query}')
    sv_rag.invoke({
        'query': query,
        'candidates': [],
        'top_docs': [],
        'draft_answer': '',
        'verify_result': {},
        'final_answer': None,
        'needs_review': False,
    })

print(f'\n📊 pass_rate={supervisor.pass_rate:.2f}，待审={len(supervisor.review_queue)} 条')
supervisor.run_hitl_review()
if supervisor.decisions:
    print('决策记录：', supervisor.decisions)


## 调试清单

按顺序排查，别一上来就改 prompt：

1. **输入** — query 是否为空？编码是否正常？
2. **上下文** — `docs` 真的被带进下一环了吗？
3. **Schema** — `Message.payload` 的字段名上下游是否一致？
4. **工具调用** — 异常是否被 `except` 静默吞掉了？
5. **Verifier** — 是否把「答非所问」判成了 pass（误判率）？
6. **重试** — 同一个 job 是否被反复处理？
7. **队列** — 是否积压？哪个 stage 最慢？

> 卡住时优先看三样东西：**trace、指标、中间输出**。

## 评估标准

| 维度 | 看什么 | 权重 |
| --- | --- | --- |
| 正确性 | 结果对不对 | ★★★★★ |
| 稳定性 | 失败后能否恢复 | ★★★★ |
| 性能 | 延迟与吞吐是否合理 | ★★★ |
| 成本 | token 与资源是否可控 | ★★★ |
| 可维护性 | 别人能否接手 | ★★★★ |

> 这五项比「看起来很聪明」更重要。

## 提交要求

- 三个练习的代码与运行输出
- 一段 200 字以内的反思：你的系统在哪个环节最脆弱？为什么？
- （可选）把练习 C 的 `decisions` 导出成 CSV，作为后续优化的样本集
